<a href="https://colab.research.google.com/github/kds825/education/blob/main/GPU%EC%82%AC%EC%9A%A9%EB%9F%89_%EA%B3%84%EC%82%B0%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Hugging Face
#      │
#      │ from_pretrained()
#      ▼
# ┌──────────────┐
# │     LLM      │
# │ 0.5B / 7B... │
# │ FP16/INT8/4  │ ← BitsAndBytesConfig
# └──────┬───────┘
#        │
#        ↑
#  **inputs 🎒
#        ↑
# ┌──────────────┐
# │  Tokenizer   │
# └──────▲───────┘
#        │
# "재무 분석해줘"
#        │
#        ▼
#    generate()
#        │
#        ▼
# 입력 Token + 생성 Token
#        │
#        ▼
# shape로 개수 측정
#        │
#        ▼
# Latency / tok/s / VRAM

In [6]:
import os
import gc
import csv
import time
import torch
from datetime import datetime
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [7]:
TEST_CONFIG ={
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "precision": "fp16",
    "users": 5,
    "prompt":
        """
        당신은 기업 재무분석 AI Agent입니다.

        매출액은 1,000억원이고
        영업이익은 전년 100억원에서
        당기 50억원으로 감소했습니다.

        주요 재무위험을 분석해주세요.
        """,
    "max_new_tokens": 100,
    "repeat": 3
}

In [8]:
RESULT_FILE = "gpu_benchmark_results.csv"
def cleanup_gpu():
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()


def load_model(
    model_name,
    precision
):
    print("===================================")
    print("모델 Loading")
    print("===================================")

    print("Model :", model_name)
    print("Precision :", precision)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token

    if precision == "fp16":
      model = AutoModelForCausalLM.from_pretrained(
          model_name,
          torch_dtype=torch.float,
          device_map = "auto"

      )

    elif precision == "int8":
      quant_config = BitsAndBytesConfig(
          load_in_8bit = True
      )
      model = AutoModelForCausalLM.from_pretrained(
          model_name,
          quantization_config=quant_config,
          device_map ="auto"
      )

    elif precision =="int4":
      quant_config = BitsAndBytesConfig(
          load_in_4bit = True
      )
      model = AutoModelForCausalLM.from_pretrained(
          model_name,
          quantization_config=quant_config,
          device_map="auto"
      )
    else:
      raise ValueError(
          "precision은 fp16 / int8 / int4 중 하나여야 합니다."
      )
    return tokenizer, model

def benchmark(config):
  model_name = config["model_name"]
  precision = config["precision"]
  users = config["users"]
  prompt = config["prompt"]
  max_new_tokens = config["max_new_tokens"]
  repeat = config["repeat"]

  if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU가 없습니다. GPU Runtime을 켜주세요"
    )
  gpu_name = torch.cuda.get_device_name(0)
  total_vram_gb = (
      torch.cuda.get_device_properties(0).total_memory
      /
      1024**3
  )

  print()
  print("===================================")
  print("GPU 정보")
  print("===================================")

  print("GPU :", gpu_name)
  print(
      "Total VRAM :",
      round(total_vram_gb, 2),
      "GB"
  )

  tokenizer, model = load_model(
      model_name,
      precision
  )

  prompts = [
      prompt
  ] * users

  inputs = tokenizer(
      prompts,
      return_tensors = "pt",
      padding = True
  )

  input_tokens_per_user = (
      inputs["input_ids"].shape[1]
  )

  inputs = inputs.to(model.device)

  latency_results = []
  token_speed_results = []
  throughput_results =[]
  peak_vram_results=[]

  for run in range(repeat):
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start = time.time()
    with torch.no_grad():
      outputs =model.generate(
          **inputs,
          max_new_tokens = max_new_tokens,
          do_sample=False,
          pad_token_id = tokenizer.pad_token_id
      )

    torch.cuda.synchronize()

    end = time.time()

    latency = end-start
    total_generated_tokens=(
        outputs.shape[1]
        -
        inputs["input_ids"].shape[1]
    ) * users

    tokens_per_second=(
        total_generated_tokens
        /
        latency
    )

    request_per_second = users / latency

    peak_vram_gb=(
        torch.cuda.max_memory_allocated()
        /
        1024**3

    )

    latency_results.append(
        latency
    )

    token_speed_results.append(
        tokens_per_second
    )
    throughput_results.append(
        request_per_second
    )
    peak_vram_results.append(
        peak_vram_gb
    )

    print()
    print(
        f"[Run {run+1}]"
    )
    print(
        "Latency:",
        round(latency,2),
        "sec"
    )
    print(
        "Token Speed:",
        round(tokens_per_second, 2),
        "tokens/sec"
    )
    print(
        "Throughput:",
        round(request_per_second, 2),
        "requests/sec"
    )
    print(
        "Peak VRAM:",
        round(peak_vram_gb, 2),
        "GB"
    )
  avg_latency = (
      sum(latency_results)
      /
      len(latency_results)
  )

  avg_token_speed=(
      sum(token_speed_results)
      /
      len(token_speed_results)
  )
  avg_throughput=(
      sum(throughput_results)
      /
      len(throughput_results)
  )
  max_peak_vram = max(
      peak_vram_results
  )

  result = {
      "timestamp":
      datetime.now().strftime(
          "%Y-%m-%d %H:%M:%S"
      ),

      "gpu": gpu_name,

      "gpu_total_vram_gb":round(total_vram_gb,2),

      "model":model_name,

      "precision":precision,

      "users":users,

      "input_tokens_per_user":input_tokens_per_user,

      "max_new_tokens":max_new_tokens,

      "repeat":repeat,

      "avg_latency_sec":round(avg_latency, 2),

      "avg_tokens_per_sec":round(avg_token_speed, 2),

      "avg_request_per_sec":round(avg_throughput, 3),

      "peak_vram_gb": round(max_peak_vram, 2)

  }

  file_exists = os.path.exists(
      RESULT_FILE
  )

  with open(
      RESULT_FILE,
      "a",
      newline="",
      encoding = "utf-8"
  ) as file:
      writer = csv.DictWriter(
          file,
          fieldnames=result.keys()
      )
      if not file_exists:
        writer.writeheader()

      writer.writerow(result)

  print()
  print("===================================")
  print("BENCHMARK RESULT")
  print("===================================")

  for key, value in result.items():
    print(
        key,
        ":",
        value
    )

  print()
  print("결과 저장 완료 →", RESULT_FILE)

  del model
  cleanup_gpu()
  return result


In [11]:
result = benchmark(TEST_CONFIG)

RuntimeError: GPU가 없습니다. GPU Runtime을 켜주세요